# Kuunda Programu za Uumbaji Picha (Azure OpenAI)

Kuna zaidi juu ya LLM kuliko uumbaji wa maandishi. Pia unaweza kuunda picha kutoka kwa maelezo ya maandishi. Picha kama sehemu ni muhimu katika MedTech, usanifu, utalii, maendeleo ya michezo, masoko, na zaidi. Katika somo hili tunatumia modeli za **GPT Image** za leo kwenye Microsoft Foundry.

## Malengo ya Kujifunza

Mwisho wa somo hili utaweza:

- Elezea ni nini uumbaji wa picha na ni wapi unafaa.
- Elewa familia ya modeli ya `gpt-image` na jinsi inavyotofautiana na modeli za zamani za DALL·E.
- Tengeneza programu ya uumbaji picha na hakikisha picha.

## Uumbaji wa picha ni nini?

Modeli za uumbaji picha huunda picha kutoka kwa maelezo ya maandishi. Modeli za kisasa kama `gpt-image` hujifunza uhusiano kati ya maandishi na picha wakati wa mafunzo, kisha kwa hatua kwa hatua hubadilisha kelele za nasibu kuwa picha inayolingana na maelezo yako.

Familia mbili maarufu za modeli za picha ni:

- **`gpt-image` (OpenAI)** — kizazi cha sasa kinachotumiwa katika somo hili. Kinaunga mkono uumbaji wa picha kutoka maandishi na uhariri wa picha (upigaji rangi ndani ya kasha).
- **Midjourney** — modeli maarufu ya mtu wa tatu yenye huduma yake na mtiririko wa kazi wa Discord.

> Modeli za picha za zamani za OpenAI — **DALL·E 2** na **DALL·E 3** — ni za urithi. DALL·E 3 haipatikani tena kwa usambazaji mpya, na vipengele kama `create_variation` vilikuwepo tu katika DALL·E 2. Tumia modeli za `gpt-image` kwa programu mpya.

Katika Microsoft Foundry, **`gpt-image-2`** ni modeli ya picha mpya kabisa na yenye uwezo mkubwa zaidi na ndiyo chaguo la msingi lililopendekezwa. `gpt-image-1.5` na `gpt-image-1-mini` pia zinapatikana kwa ujumla.

> **Muhimu:** modeli za `gpt-image` hurudisha picha iliyoundwa kama **base64** (`b64_json`), sio kama URL. Msimbo wako hutafsiri mfuatano wa base64 kwa biti na kuihifadhi — hakuna URL ya picha ya kupakua.


## Kujenga programu yako ya kwanza ya uzalishaji picha

Kwa hivyo ni nini kinachohitajika kujenga programu ya uzalishaji picha? Unahitaji maktaba zifuatazo:

- **python-dotenv**, unashauriwa sana kutumia maktaba hii kuweka siri zako katika faili la *.env* mbali na msimbo.
- **openai**, maktaba hii ndio utakayoiweka kutumia kuwasiliana na API ya OpenAI.
- **pillow**, kwa kufanya kazi na picha ndani ya Python.

Ikiwa bado hujafanya, fuata maagizo yaliyo kwenye ukurasa wa [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) kuunda rasilimali ya Microsoft Foundry na mfano. Chagua **gpt-image-2** kama mfano (mfano wa picha wa Azure OpenAI wa hivi karibuni; DALL·E ni urithi).

1. Unda faili la *.env* lenye maudhui yafuatayo:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Tafuta habari hii katika lango la Microsoft Foundry kwa rasilimali yako katika sehemu ya "Deployments".



1. Kusanya maktaba zilizo juu kwenye faili linaloitwa *requirements.txt* kama ifuatavyo:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Ifuatayo, tengeneza mazingira pepe na usakinishe maktaba:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Kwa Windows, tumia amri zifuatazo kuunda na kuzindua mazingira yako pepe:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Ongeza msimbo ufuatao kwenye faili liitwalo *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # ingiza dotenv
    dotenv.load_dotenv()

    # sanidi mteja wa Azure OpenAI (Microsoft Foundry).
    # Mifano ya picha inahitaji toleo jipya la API — angalia nyaraka za Foundry kwa lile ambalo mfano wako unahitaji.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Unda picha kwa kutumia API ya uzalishaji wa picha
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Weka maandishi ya maelekezo yako hapa
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # kwa mfano "gpt-image-2"
        )
        # Weka saraka ya picha iliyohifadhiwa
        image_dir = os.path.join(os.curdir, 'images')

        # Ikiwa saraka haipo, tengeneza
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Anzisha njia ya picha (kumbuka aina ya faili inapaswa kuwa png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # mifano ya gpt-image hurudisha picha kama base64 (b64_json), si URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Onyesha picha kwenye mtazamaji wa picha wa chaguo-msingi
        image = Image.open(image_path)
        image.show()

    # shika makosa
    except BadRequestError as err:
        print(err)

    ```

Hebu elezea msimbo huu:

- Kwanza, tunaingiza maktaba tunazohitaji, ikiwemo maktaba ya OpenAI, maktaba ya dotenv, moduli ya base64, na maktaba ya Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Ifuatayo, tunapakua vigezo vya mazingira kutoka kwa faili *.env*.

    ```python
    # ingiza dotenv
    dotenv.load_dotenv()
    ```

- Baadaye, tunaandaa mteja wa Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Ifuatayo, tunatengeneza picha:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Ingiza maandishi yako ya agizo hapa
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` mifano hurejesha picha kama kamba ya **base64** katika `data[0].b64_json`. Tunaitafsiri kuwa biti na kuandika kwenye faili — hakuna URL ya kupakua.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Mwishowe, tunafungua picha na kutumia hadhira ya picha ya kawaida kuionyesha:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Maelezo zaidi juu ya kutengeneza picha

Tazama vigezo vya `images.generate`:

- **prompt**, ni maandishi ya mwongozo yanayotumika kutengeneza picha. Hapa ni "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, ni ukubwa wa picha inayotengenezwa (kwa mfano `1024x1024`, `1536x1024`, `1024x1536`, au `"auto"`).
- **n**, ni idadi ya picha zinazotengenezwa. Hapa tunatengeneza moja.
- **model**, ni jina la usambazaji wa mfano wako wa picha (kwa mfano `gpt-image-2`).

> Mifano ya picha haisikii kigezo cha `temperature` — hicho ni udhibiti wa uundaji wa maandishi. Ili kupata utofauti, piga API tena; kupunguza utofauti, fanya mwongozo wako kuwa maalum zaidi.

## Uwezo wa ziada wa uundaji wa picha

Umeona jinsi ya kutengeneza picha kwa mistari michache ya Python. Mifano ya `gpt-image` pia inaweza **kurekebisha** picha iliyopo. Kwa kutoa picha, **mask** ya hiari (inayoashiria eneo la kubadilisha), na mwongozo, unaweza kubadilisha sehemu ya picha — kwa mfano, ongeza kofia kwa sungura wetu.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# mabadiliko pia hurudishwa kama base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Picha ya msingi ina sungura tu; picha ya mwisho ina kofia kwenye sungura.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
